# Build NB for MOFA Approach

2026.03.24 
Built by Bryant A. Chambers
Built for Aegis

## Background

Goal: Identify Modules in aeDNA that are driven by multiple levels of prior information. Two approaches are tested in the Network project, PLIER which relies on one level of prior information but collapses the exposure environment into only the prior information and the testing samples and , two, MOFA, which optimizes mutiple levels at once. The two central methods rely on matrix factorization. PLIER uses an older approach: 


$$
Y = CUB + \Delta B + E
$$

Here the assumption is made that there is some matrix, Y, that contains the "expression" of genes in each sample. This matrix can be summarized as loadings B and Z, that is Y = BZ, but the loadings in Z can be summarized as UC so you get Y = CUB with Error. This allows us to take prior information like mappings to KEGG orthologs and identify how much they might contribute to the expression of genes in each of the samples or in our case, the overall metabolism occuring in each of the modules!!! So if we use prior information from KEGG or BGCs we can start to understand the microbiome in this context rather than just correlational networking. This *might* drive module fomation into functional units more easily.

The issue here is that the data is collapsed into the Latent Value representation and would also need some amount of Feature Set Enrichment Analysis (FSEA) on the latent factor loadings.
Instead of doing a standard enrichment on a list of differentially abundant genes, you take the loadings (weights) of every species or KO in a specific Latent Factor and run a GSEA-style test.

This identifies whether a specific Latent Factor is "enriched" for a certain KEGG Pathway or a certain Taxonomic Class. This is the "Species enrichment per latent variable" you're looking for.

A more modern version of this, with bayesian factor analysis, would allow Multiview of mutliple elements of prior knowledge which solves this problem without needing FSEA. MOFA assumes a power of loading from multiple loadings.

$$
Y_m = Z(W_m)^T + \epsilon_m
$$

$Y_m$
 : Your actual data matrix for a specific view m (e.g., your OTU table).

Z: The Latent Factors (The hidden biological drivers operating across all samples). This is a single matrix shared across all views.

$W_m$
 : The Weights (How strongly each specific feature—like a specific KEGG pathway or a specific BGC—is connected to the latent factor). Each view gets its own weight matrix.

$ϵ_m$
 : The residual noise (what the model couldn't explain).

The Goal: MOFA calculates Z and $W_m$
  to explain as much variance in $Y_m$
  as possible.

**Data needed for Study**  
The Data Preparation (Common Formats)  
Before feeding data to the model, it must be formatted and normalized so that one highly abundant view doesn't drown out the others.  
    - View 1: OTUs/ASVs (Taxonomy):  
     Format: Rows = Samples, Columns = Taxa. 
     Prep: Raw counts should be transformed. Centered Log-Ratio (CLR) transformation is the gold standard here to handle the compositional nature of sequencing data.  
    - View 2: KEGG (Functional Potential):  
    Format: Rows = Samples, Columns = KOs or Pathways.  
    Prep: Usually Log2-transformed Transcripts Per Million (TPM) or similar normalized abundances.  
    - View 3: BGCs (Secondary Metabolites):  
    Format: Rows = Samples, Columns = Gene Clusters (e.g., from antiSMASH or BiG-SCAPE).  
    Prep: Binary (presence/absence) or normalized count data. MOFA can handle different statistical distributions (e.g., Poisson for counts, Bernoulli for binary).  
    - Metadata (The Environment):  
    Format: Rows = Samples, Columns = Covariates (Sediment Depth, Site Location, Paleoecosystem classification).  

Note: Metadata is not fed into the initial MOFA training. It is kept aside for the interpretation step.

## Synthetic Data

In [2]:
# Load transformation library
# install.packages("compositions")
library(compositions)

set.seed(42) # For reproducibility


Welcome to compositions, a package for compositional data analysis.
Find an intro with "? compositions"



Attaching package: ‘compositions’


The following object is masked from ‘package:igraph’:

    normalize


The following objects are masked from ‘package:stats’:

    anova, cor, cov, dist, var


The following object is masked from ‘package:graphics’:

    segments


The following objects are masked from ‘package:base’:

    %*%, norm, scale, scale.default




In [3]:

# 1. Setup Experimental Design
# ------------------------------------------------------------
n_samples <- 50
sample_names <- paste0("Sample_", 1:n_samples)

# Create a "Hidden Signal" (e.g., a Sediment Depth gradient)
# This is the "Latent Factor" we hope MOFA will rediscover.
depth_signal <- seq(-2, 2, length.out = n_samples) 

# 2. View 1: Taxonomy (mat_taxa)
# ------------------------------------------------------------
# We'll simulate 100 Genera. 
# Some will increase with depth, some will decrease, others are noise.
n_taxa <- 100
mat_taxa_raw <- matrix(rpois(n_samples * n_taxa, lambda = 10), 
                       nrow = n_taxa, ncol = n_samples)

# Inject signal into the first 20 taxa
for(i in 1:20) {
  mat_taxa_raw[i, ] <- rpois(n_samples, lambda = exp(2 + (depth_signal * runif(1, -1, 1))))
}

rownames(mat_taxa_raw) <- paste0("Genus_", 1:n_taxa)
colnames(mat_taxa_raw) <- sample_names

# CRITICAL STEP: CLR Transformation
# We add a small pseudocount (1) to handle zeros, then CLR.
mat_taxa <- t(clr(t(mat_taxa_raw + 1)))

In [6]:
View(mat_taxa_raw %>% head(10))
View(mat_taxa %>% head(10)) 

,Sample_1,Sample_2,Sample_3,Sample_4,Sample_5,Sample_6,Sample_7,Sample_8,Sample_9,Sample_10,⋯,Sample_41,Sample_42,Sample_43,Sample_44,Sample_45,Sample_46,Sample_47,Sample_48,Sample_49,Sample_50
Genus_1,2,5,5,5,4,6,9,6,8,6,⋯,9,16,7,9,10,6,11,13,11,12
Genus_2,8,7,6,9,7,8,6,6,5,6,⋯,7,16,9,9,11,13,13,11,14,15
Genus_3,0,0,2,1,1,3,1,0,0,1,⋯,19,33,27,31,39,35,37,38,62,55
Genus_4,46,47,41,30,31,33,32,33,19,27,⋯,4,3,3,3,1,2,4,1,2,0
Genus_5,2,2,0,0,2,4,4,1,3,1,⋯,20,15,21,16,24,18,36,31,20,29
Genus_6,3,6,5,5,4,5,5,9,6,7,⋯,8,9,7,2,4,9,9,8,11,13
Genus_7,30,24,36,21,28,19,24,25,14,21,⋯,2,3,1,2,1,1,1,2,1,1
Genus_8,1,1,1,1,4,3,2,5,1,2,⋯,25,25,33,35,45,37,41,45,59,46
Genus_9,8,5,6,10,8,10,5,9,5,7,⋯,10,8,7,7,6,10,7,8,9,9
Genus_10,64,49,53,36,34,36,54,30,21,24,⋯,1,4,1,2,1,1,1,0,1,2


           Sample_1   Sample_2   Sample_3    Sample_4   Sample_5    Sample_6
Genus_1  -1.2188028 -0.4666907 -0.5048284 -0.45032617 -0.7044113 -0.34925982
Genus_2  -0.1201905 -0.1790087 -0.3506777  0.06049945 -0.2344077 -0.09794539
Genus_3  -2.3174151 -2.2584502 -1.1979756 -1.54893846 -1.6207020 -0.90887561
Genus_4   1.5327325  1.6127508  1.4410818  1.19190156  1.1518867  1.23119056
Genus_5  -1.2188028 -1.1598379 -2.2965879 -2.24208564 -1.2152369 -0.68573205
Genus_6  -0.9311207 -0.3125401 -0.5048284 -0.45032617 -0.7044113 -0.50341050
Genus_7   1.1165721  0.9604256  1.3143301  0.84895681  1.0534466  0.70056231
Genus_8  -1.6242679 -1.5653030 -1.6034407 -1.54893846 -0.7044113 -0.90887561
Genus_9  -0.1201905 -0.4666907 -0.3506777  0.15580963 -0.1166246  0.10272531
Genus_10  1.8569722  1.6535728  1.6923962  1.36883227  1.2414988  1.31574795
            Sample_7    Sample_8    Sample_9  Sample_10  Sample_11   Sample_12
Genus_1  -0.03369764 -0.41880150 -0.06842691 -0.3582266 -0.3590098  0.2575

In [8]:
#View 2: Metabolism (mat_kegg)
# ------------------------------------------------------------
# 150 KEGG Orthologs (KOs). Let's assume these are continuous (e.g., TPM).
n_kegg <- 150
mat_kegg <- matrix(rnorm(n_samples * n_kegg, mean = 5, sd = 1), 
                   nrow = n_kegg, ncol = n_samples)

# Inject signal: KOs linked to the taxa above
for(i in 1:30) {
  mat_kegg[i, ] <- 5 + (depth_signal * rnorm(1, 2, 0.5)) + rnorm(n_samples, sd = 0.5)
}

rownames(mat_kegg) <- paste0("KO_", sprintf("%05d", 1:n_kegg))
colnames(mat_kegg) <- sample_names

In [9]:
View((mat_kegg %>% head(10)))

,Sample_1,Sample_2,Sample_3,Sample_4,Sample_5,Sample_6,Sample_7,Sample_8,Sample_9,Sample_10,⋯,Sample_41,Sample_42,Sample_43,Sample_44,Sample_45,Sample_46,Sample_47,Sample_48,Sample_49,Sample_50
KO_00001,-1.29972941,-0.78754382,-0.2906005,0.1165054,-0.30423196,1.1858703,0.7778557,0.9604310,1.4140838,1.415266,⋯,9.089485,8.428235,8.237438,9.945881,10.216619,9.396183,10.107370,9.799780,10.043800,10.487718
KO_00002,1.45586764,1.88512444,1.3719790,0.8323012,1.03772734,1.2656904,1.0380149,1.6825165,1.9933254,2.297644,⋯,7.651731,7.623662,8.099840,8.558710,8.220797,9.300055,8.716171,8.411036,8.510065,9.522556
KO_00003,-0.49953660,0.03267046,-0.9305171,0.4127592,-0.08518338,0.4118247,1.4273092,0.8066311,1.9254321,1.106274,⋯,8.923576,8.293517,9.582951,8.089580,10.017837,8.585781,8.672946,10.231039,9.470912,9.701307
KO_00004,2.54456586,1.82513958,2.1768062,2.6150445,2.14916345,2.9111509,2.7276520,2.9351830,3.7832299,3.368196,⋯,6.737643,6.367823,7.749519,7.865110,7.387896,8.011373,7.021159,7.485900,8.402905,8.287148
KO_00005,0.62629751,0.99900191,1.3427620,0.5921970,1.25135915,1.3697492,1.4219169,2.0240760,1.7148818,3.096745,⋯,7.580416,7.660809,9.268584,8.134700,7.525093,8.622754,9.982642,9.190569,9.184468,9.781480
KO_00006,0.30298912,1.13577552,1.5930158,1.4578158,1.61018232,2.8200756,2.5803228,2.3366107,2.6866324,2.204615,⋯,8.022745,7.019111,7.897333,8.792210,8.034764,7.544398,9.084657,8.340401,8.861262,9.008078
KO_00007,1.13594704,1.01572048,1.7538967,1.6942289,0.81841617,2.7944718,2.3013059,2.5741866,2.1836889,2.794719,⋯,7.008761,8.218077,7.408145,8.099806,8.275021,8.116633,9.712045,8.086330,8.744828,9.206783
KO_00008,0.01466556,0.14627593,0.3661152,-0.3591215,0.25446255,1.1324444,0.7387440,1.8380997,0.9771405,0.652335,⋯,7.718560,8.397797,9.458594,8.924907,9.330327,9.162784,9.520518,10.262447,10.237585,10.875601
KO_00009,1.54705700,1.53165473,0.4590590,0.5047266,1.35030823,2.3880819,1.6819395,1.4153552,2.5868106,1.823657,⋯,8.046654,8.123137,7.545678,7.848539,8.558875,9.505953,8.200361,9.054620,9.825381,7.806405
KO_00010,1.14845500,1.45220580,0.6311613,1.2241819,2.38490941,1.3035340,1.8635593,1.9609274,2.9348601,1.930789,⋯,6.735260,7.348544,8.280673,8.191986,7.850820,8.112064,8.561257,7.974516,8.839303,9.316379


In [10]:
# 4. View 3: Biosynthesis (mat_bgc)
# ------------------------------------------------------------
# 50 BGCs. We'll use binary presence/absence (Bernoulli distribution).
n_bgc <- 50
mat_bgc <- matrix(rbinom(n_samples * n_bgc, 1, prob = 0.2), 
                  nrow = n_bgc, ncol = n_samples)

# Inject signal: Certain BGCs appear only at specific depths
for(i in 1:10) {
  prob_signal <- 1 / (1 + exp(-(depth_signal * rnorm(1, 2, 0.5)))) # Logistic mapping
  mat_bgc[i, ] <- rbinom(n_samples, 1, prob = prob_signal)
}

rownames(mat_bgc) <- paste0("BGC_Cluster_", 1:n_bgc)
colnames(mat_bgc) <- sample_names

In [11]:
mat_bgc %>% head(10)

,Sample_1,Sample_2,Sample_3,Sample_4,Sample_5,Sample_6,Sample_7,Sample_8,Sample_9,Sample_10,⋯,Sample_41,Sample_42,Sample_43,Sample_44,Sample_45,Sample_46,Sample_47,Sample_48,Sample_49,Sample_50
BGC_Cluster_1,0,0,0,0,0,0,0,0,0,1,⋯,1,1,1,1,1,1,1,1,1,1
BGC_Cluster_2,0,0,0,0,0,0,1,0,1,0,⋯,1,1,1,0,0,1,1,1,1,1
BGC_Cluster_3,0,0,0,0,0,0,0,0,0,0,⋯,1,1,0,1,1,1,1,1,1,1
BGC_Cluster_4,0,0,0,0,0,0,0,0,0,1,⋯,1,1,1,1,1,1,1,1,1,1
BGC_Cluster_5,0,0,0,0,0,0,0,0,0,0,⋯,1,1,1,1,1,1,1,1,1,1
BGC_Cluster_6,0,0,0,0,0,0,0,0,1,0,⋯,1,1,1,1,1,1,1,1,1,1
BGC_Cluster_7,0,0,0,0,0,0,1,0,0,0,⋯,0,1,1,1,1,1,1,1,1,1
BGC_Cluster_8,0,0,0,0,0,0,0,0,0,0,⋯,1,1,1,1,1,1,1,1,1,1
BGC_Cluster_9,0,0,0,0,0,0,1,1,0,1,⋯,1,1,1,1,1,1,1,1,1,1
BGC_Cluster_10,0,0,0,0,0,0,0,0,1,0,⋯,1,1,1,1,1,1,1,1,1,1


In [12]:
data_list <- list(
  Taxonomy = mat_taxa,
  Metabolism = mat_kegg,
  Biosynthesis = mat_bgc
)

In [35]:
data_list_analysis <- list(
  Taxonomy = mat_taxa %>% unname() %>% as.matrix(),
  Metabolism = mat_kegg %>% unname() %>% as.matrix(),
  Biosynthesis = mat_bgc %>% unname() %>% as.matrix()
)

In [34]:
mat_taxa %>% unname() %>% as.matrix()

-1.21880277,-0.46669074,-0.504828384,-0.45032617,-0.70441130,-0.349259818,-0.03369764,-0.41880150,-0.06842691,-0.358226581,⋯,-0.0003131443,0.47595810,-0.23735909,-0.05701616,0.03786519,-0.39142323,0.10193504,0.35497558,0.12880751,0.29443385
-0.12019049,-0.17900867,-0.350677705,0.06049945,-0.23440767,-0.097945390,-0.39037258,-0.41880150,-0.47389202,-0.358226581,⋯,-0.2234566956,0.47595810,-0.01421554,-0.05701616,0.12487656,0.30172395,0.25608572,0.20082490,0.35195106,0.50207322
-2.31741506,-2.25845021,-1.197975565,-1.54893846,-1.62070204,-0.908875606,-1.64313555,-2.36471165,-2.26565149,-1.610989549,⋯,0.6928340363,1.16910528,1.01540388,1.10613465,1.32884937,1.24618556,1.25461455,1.37947989,1.78703558,1.75483619
1.53273254,1.61275080,1.441081765,1.19190156,1.15188669,1.231190557,1.16022483,1.16164887,0.73008079,1.028067780,⋯,-0.6934603248,-0.97096088,-0.93050627,-0.97330689,-1.66688291,-1.23872109,-0.77353369,-1.59093457,-1.25748685,-2.27051550
-1.21880277,-1.15983792,-2.296587854,-2.24208564,-1.21523693,-0.685732055,-0.72684482,-1.67156447,-0.87935712,-1.610989549,⋯,0.7416242004,0.41533348,0.77424183,0.47361209,0.85884574,0.60710560,1.22794631,1.18165415,0.68842329,1.13068188
-0.93112070,-0.31254006,-0.504828384,-0.45032617,-0.70441130,-0.503410498,-0.54452326,-0.06212656,-0.31974134,-0.224695188,⋯,-0.1056736599,-0.05467015,-0.23735909,-1.26098896,-0.75059217,-0.03474829,-0.08038651,-0.08685717,0.12880751,0.36854183
1.11657214,0.96042562,1.314330059,0.84895681,1.05344661,0.700562306,0.88259310,0.89338488,0.44239872,0.786905723,⋯,-1.2042859486,-0.97096088,-1.62365345,-1.26098896,-1.66688291,-1.64418620,-1.68982443,-1.18546946,-1.66295196,-1.57736832
-1.62426788,-1.56530303,-1.603440673,-1.54893846,-0.70441130,-0.908875606,-1.23767044,-0.57295218,-1.57250430,-1.205524441,⋯,0.9551983007,0.90084129,1.20955990,1.22391769,1.46861131,1.30025278,1.35469801,1.54455964,1.73824542,1.57963210
-0.12019049,-0.46669074,-0.350677705,0.15580963,-0.11662464,0.102725305,-0.54452326,-0.06212656,-0.47389202,-0.224695188,⋯,0.0949970355,-0.16003067,-0.23735909,-0.28015971,-0.41411994,0.06056189,-0.30353006,-0.08685717,-0.05351405,0.03206959
1.85697221,1.65357280,1.692396193,1.36883227,1.24149885,1.315747945,1.67105046,1.06927555,0.82539097,0.914739095,⋯,-1.6097510567,-0.74781733,-1.62365345,-1.26098896,-1.66688291,-1.64418620,-1.68982443,-2.28408175,-1.66295196,-1.17190322
0.32164227,0.22645644,0.188318796,0.15580963,0.08404606,0.343887362,0.30277460,-0.28527011,-0.31974134,-0.358226581,⋯,-0.6934603248,0.12765140,-0.52504116,-0.16237667,-0.56827062,-0.54557391,-0.30353006,-0.89778739,-0.41018899,-0.47875603


## MOFA Analysis



In [43]:
library(MOFA2)
library(dplyr)
library(tidyr)
library(igraph)
library(reticulate)

#reticulate::use_condaenv("amod_mofapy2")
reticulate::use_python("/maps/projects/caeg/people/gfx654/miniforge3/envs/amod_mofapy2/bin/python", required = TRUE)

In [ ]:
lapply(data_list_analysis, View)

In [44]:
MOFAobject <- create_mofa(data_list_analysis)


Creating MOFA object from a list of matrices (features as rows, sample as columns)...




Warning message in create_mofa_from_matrix(data, groups):
“Feature names are not specified for view 1, using default: feature1_v1, feature2_v1...”
Warning message in create_mofa_from_matrix(data, groups):
“Feature names are not specified for view 2, using default: feature1_v2, feature2_v2...”
Warning message in create_mofa_from_matrix(data, groups):
“Feature names are not specified for view 3, using default: feature1_v3, feature2_v3...”
Warning message in create_mofa_from_matrix(data, groups):
“Sample names for group 1 are not specified, using default: sample1_g1, sample2_g1,...”


In [45]:
# Set Data Options: Tell MOFA what kind of statistical distributions to expect.
# We use "gaussian" for continuous/normalized data.
data_opts <- get_default_data_options(MOFAobject)
data_opts$scale_views <- TRUE # Scales views so one doesn't dominate the model


In [46]:

# Set Model Options: How many factors (hidden drivers) should it look for?
model_opts <- get_default_model_options(MOFAobject)
model_opts$num_factors <- 10 # A good starting point. MOFA will drop inactive ones.


In [47]:

# Set Training Options
train_opts <- get_default_training_options(MOFAobject)
train_opts$convergence_mode <- "medium" 

# Prepare and Train (This connects to Python's mofapy2 under the hood)
MOFAobject <- prepare_mofa(MOFAobject, data_opts, model_opts, train_opts)
MOFAobject <- run_mofa(MOFAobject, use_basilisk = FALSE)


Checking data options...

Checking training options...

Checking model options...

Warning message in run_mofa(MOFAobject, use_basilisk = FALSE):
“No output filename provided. Using /tmp/Rtmp0ncwDu/mofa_20260324-112628.hdf5 to store the trained model.

”
Connecting to the mofapy2 python package using reticulate (use_basilisk = FALSE)... 
    Please make sure to manually specify the right python binary when loading R with reticulate::use_python(..., force=TRUE) or the right conda environment with reticulate::use_condaenv(..., force=TRUE)
    If you prefer to let us automatically install a conda environment with 'mofapy2' installed using the 'basilisk' package, please use the argument 'use_basilisk = TRUE'


8 factors were found to explain no variance and they were removed for downstream analysis. You can disable this option by setting load_model(..., remove_inactive_factors = FALSE)

Warning message in .quality_control(object, verbose = verbose):
“Factor(s) 1 are strongly correlated wit

In [48]:

# 4. Interpret and Select a Factor
# ------------------------------------------------------------------------------
# In a real workflow, you would correlate the factors with your metadata here.
# e.g., Correlate Factor scores with "Sediment Depth" or "Paleoecosystem State".
# Let's assume we found that Factor 3 strongly correlates with a key environmental shift.
target_factor <- "Factor1"


In [49]:

# 5. Extract Weights to Build the Network (Node & Edge Lists)
# ------------------------------------------------------------------------------
# We extract the weights (loadings) for all features across all views for Factor 3.
weights <- get_weights(MOFAobject, factors = target_factor, as.data.frame = TRUE)

# Filter for the most important features (the "signal").
# We take the absolute value of the weight (magnitude matters, sign dictates direction).
# Here, we keep features in the top 5% of weights for this specific factor.
threshold <- quantile(abs(weights$value), 0.95)
top_features <- weights %>% filter(abs(value) >= threshold)

# Create Node List
# We need to know what view each feature came from so Cytoscape can color them differently.
nodes <- top_features %>%
  select(id = feature, type = view, weight = value) %>%
  distinct()

# Create Edge List
# How do we connect them? In MOFA, if two features are highly weighted on the SAME factor,
# they are co-varying. We create edges between all top features in this factor.
# Note: This creates a fully connected sub-graph (clique) for this factor. 
# For multi-factor networks, you would calculate pairwise correlations of their weight profiles.
edges <- expand.grid(source = nodes$id, target = nodes$id) %>%
  filter(source != target) %>% # Remove self-loops
  # Optional: Remove duplicate undirected edges (A-B and B-A)
  mutate(combo = paste(pmin(source, target), pmax(source, target), sep="_")) %>%
  distinct(combo, .keep_all = TRUE) %>%
  select(source, target)

# Add an edge attribute (e.g., they belong to the Factor 3 interaction module)
edges$module <- target_factor

# 6. Build and Export for Cytoscape
# ------------------------------------------------------------------------------
# Create the igraph object
network_graph <- graph_from_data_frame(d = edges, vertices = nodes, directed = FALSE)

# Export to GraphML (Cytoscape's preferred format)
# In Cytoscape, simply go to File -> Import -> Network from File -> Select this GraphML.
write_graph(network_graph, "MOFA_Factor3_Network.graphml", format = "graphml")

# Alternatively, export clean CSVs for custom knowledge graph ingestion
write.csv(nodes, "MOFA_Node_List.csv", row.names = FALSE)
write.csv(edges, "MOFA_Edge_List.csv", row.names = FALSE)

print("Network construction complete. Ready for Cytoscape or Knowledge Graph integration.")

Warning message:
“There were 2 warnings in `mutate()`.
The first warning was:
ℹ In argument: `combo = paste(pmin(source, target), pmax(source, target), sep =
  "_")`.
Caused by warning in `Ops.factor()`:
! ‘>’ not meaningful for factors
ℹ Run `dplyr::last_dplyr_warnings()` to see the 1 remaining warning.”


[1] "Network construction complete. Ready for Cytoscape or Knowledge Graph integration."
